
# Analyze the near-null PCA constraint coordinate $C$

This notebook computes and analyzes the approximate constraint coordinate

\[
C = \sum_j w_{5j} z_j,
\]

where \(w_{5j}\) are the PC5 loadings and \(z_j\) are standardized transformed force-constant features:

\[
(\alpha_0,\; \alpha_1+2\beta_1,\; \alpha_1-\beta_1,\; \alpha_2,\; \beta_2).
\]

Because PC5 has nearly zero explained variance, \(C\) measures the small displacement perpendicular to the dominant four-dimensional force-constant manifold. The goal is to test whether \(C\) is purely numerical, or whether it correlates with stability, fitness, family membership, or phonon properties.

The notebook:

1. Loads `dataframe000013.pkl` through `dataframe000016.pkl`.
2. Selects the top `TOP_K` solutions per `(dataset, mass, a)` ranked by `fitness_norm`.
3. Constructs the transformed coordinates.
4. Recomputes PCA and defines \(C\equiv s_5\).
5. Saves publication-quality figures and CSV summaries.


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%config InlineBackend.figure_format = "retina"

# -------------------------------------------------------------------
# User settings
# -------------------------------------------------------------------
DATA_DIR = Path("./dataframes/")  # Update this path if needed.

DATAFRAME_FILES = {
    "dataframe000013": DATA_DIR / "dataframe000013.pkl",
    "dataframe000014": DATA_DIR / "dataframe000014.pkl",
    "dataframe000015": DATA_DIR / "dataframe000015.pkl",
    "dataframe000016": DATA_DIR / "dataframe000016.pkl",
}

TOP_K = 5
RANK_BY = "fitness_norm"

OUTPUT_DIR = Path("pca_constraint_coordinate_C_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FORMATS = ("pdf", "png")
DPI = 300
FIGSIZE = (6.4, 4.8)
FIGSIZE_WIDE = (7.2, 4.8)

# Family threshold for qualitative classification.
# r_alpha2 = |alpha2|/(|alpha2| + |beta2|)
# near 0: beta2 dominated; near 1: alpha2 dominated.
FAMILY_LOW = 0.33
FAMILY_HIGH = 0.67


In [ ]:

def find_first_existing(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


def load_dataframe(path, dataset_label):
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Update DATA_DIR or DATAFRAME_FILES."
        )
    df = pd.read_pickle(path).copy()
    df["dataset"] = dataset_label
    return df


def apply_publication_axes(ax):
    ax.tick_params(
        axis="both", which="both", direction="in", top=True, right=True,
        labelsize=12, length=5, width=1.1,
    )
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)


def save_figure(fig, stem, output_dir=OUTPUT_DIR, formats=SAVE_FORMATS):
    saved = []
    for ext in formats:
        outpath = output_dir / f"{stem}.{ext}"
        fig.savefig(outpath, dpi=DPI, bbox_inches="tight")
        saved.append(outpath)
    print("Saved:")
    for path in saved:
        print(f"  {path}")
    return saved


def display_label(col):
    label_map = {
        "alpha0": r"$\alpha_0$",
        "alpha_0": r"$\alpha_0$",
        "alpha1": r"$\alpha_1$",
        "alpha_1": r"$\alpha_1$",
        "beta1": r"$\beta_1$",
        "beta_1": r"$\beta_1$",
        "alpha1_plus_2beta1": r"$\alpha_1 + 2\beta_1$",
        "alpha1_minus_beta1": r"$\alpha_1 - \beta_1$",
        "alpha2": r"$\alpha_2$",
        "alpha_2": r"$\alpha_2$",
        "beta2": r"$\beta_2$",
        "beta_2": r"$\beta_2$",
        "fitness_norm": r"$f_{\rm norm}$",
        "min_frequency": r"$\omega_{\min}$ (THz)",
        "max_frequency": r"$\omega_{\max}$ (THz)",
        "num_imaginary": r"$N_{-}$",
    }
    return label_map.get(col, col)


def classify_family(r):
    if r <= FAMILY_LOW:
        return r"$\beta_2$-dominated"
    if r >= FAMILY_HIGH:
        return r"$\alpha_2$-dominated"
    return "mixed"


In [ ]:

# -------------------------------------------------------------------
# Load and select top-k solutions
# -------------------------------------------------------------------
frames = []
for label, path in DATAFRAME_FILES.items():
    tmp = load_dataframe(path, label)
    print(f"{label}: {tmp.shape[0]:,} rows, {tmp.shape[1]:,} columns")
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"\nAggregated dataframe before filtering: {df_all.shape[0]:,} rows")

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter"], df_all.columns)
rank_col = find_first_existing([RANK_BY, "fitness_norm", "fnorm"], df_all.columns)

alpha0_col = find_first_existing(["alpha0", "alpha_0"], df_all.columns)
alpha1_col = find_first_existing(["alpha1", "alpha_1"], df_all.columns)
beta1_col  = find_first_existing(["beta1", "beta_1"], df_all.columns)
alpha2_col = find_first_existing(["alpha2", "alpha_2"], df_all.columns)
beta2_col  = find_first_existing(["beta2", "beta_2"], df_all.columns)

fitness1_col = find_first_existing(["fitness1", "f1"], df_all.columns)
fitness2_col = find_first_existing(["fitness2", "f2"], df_all.columns)
fitness3_col = find_first_existing(["fitness3", "f3"], df_all.columns)
minfreq_col = find_first_existing(["min_frequency", "min_freq", "omega_min"], df_all.columns)
maxfreq_col = find_first_existing(["max_frequency", "max_freq", "omega_max"], df_all.columns)
numimag_col = find_first_existing(["num_imaginary", "n_imaginary", "num_imag", "n_negative", "num_negative"], df_all.columns)

required = [mass_col, alat_col, rank_col, alpha0_col, alpha1_col, beta1_col, alpha2_col, beta2_col]
if any(col is None for col in required):
    raise ValueError(
        "Could not detect all required columns. "
        "Check mass, lattice parameter, rank, and force-constant column names."
    )

print("\nDetected columns:")
for name, col in {
    "mass": mass_col,
    "lattice parameter": alat_col,
    "rank": rank_col,
    "alpha0": alpha0_col,
    "alpha1": alpha1_col,
    "beta1": beta1_col,
    "alpha2": alpha2_col,
    "beta2": beta2_col,
    "fitness1": fitness1_col,
    "fitness2": fitness2_col,
    "fitness3": fitness3_col,
    "min_frequency": minfreq_col,
    "max_frequency": maxfreq_col,
    "num_imaginary": numimag_col,
}.items():
    print(f"  {name:18s}: {col}")

# Select the top TOP_K solutions per dataset, mass, and lattice parameter.
df_top = (
    df_all.sort_values(rank_col, ascending=False)
    .groupby(["dataset", mass_col, alat_col], group_keys=False)
    .head(TOP_K)
    .copy()
)

# Ensure relevant columns are numeric.
numeric_cols = [mass_col, alat_col, rank_col, alpha0_col, alpha1_col, beta1_col, alpha2_col, beta2_col]
for optional in [fitness1_col, fitness2_col, fitness3_col, minfreq_col, maxfreq_col, numimag_col]:
    if optional is not None:
        numeric_cols.append(optional)

for col in sorted(set(numeric_cols)):
    df_top[col] = pd.to_numeric(df_top[col], errors="coerce")

print(f"\nSelected top {TOP_K} per (dataset, mass, a): {df_top.shape[0]:,} rows")


In [ ]:

# -------------------------------------------------------------------
# Construct transformed force-constant coordinates and family labels
# -------------------------------------------------------------------
df_top["alpha1_plus_2beta1"] = df_top[alpha1_col] + 2.0 * df_top[beta1_col]
df_top["alpha1_minus_beta1"] = df_top[alpha1_col] - df_top[beta1_col]

df_top["abs_alpha2"] = df_top[alpha2_col].abs()
df_top["abs_beta2"] = df_top[beta2_col].abs()
df_top["r_alpha2"] = df_top["abs_alpha2"] / (df_top["abs_alpha2"] + df_top["abs_beta2"] + 1e-12)
df_top["family"] = df_top["r_alpha2"].apply(classify_family)

force_constant_cols = [
    alpha0_col,
    "alpha1_plus_2beta1",
    "alpha1_minus_beta1",
    alpha2_col,
    beta2_col,
]

print("PCA feature columns:")
for col in force_constant_cols:
    print(f"  {col:24s} -> {display_label(col)}")

print("\nFamily counts:")
print(df_top["family"].value_counts().to_string())


In [ ]:

# -------------------------------------------------------------------
# Standardize and compute PCA
# -------------------------------------------------------------------
X = df_top[force_constant_cols].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)

for i in range(scores.shape[1]):
    df_top[f"PC{i+1}"] = scores[:, i]

# The constraint coordinate C is the fifth PC score.
df_top["C"] = df_top["PC5"]
df_top["abs_C"] = df_top["C"].abs()

explained = pca.explained_variance_ratio_
eigenvalues = pca.explained_variance_
singular_values = pca.singular_values_

print("Explained variance ratio, full precision:")
for i, val in enumerate(explained, start=1):
    print(f"PC{i}: {val:.16e}")

print("\nEigenvalues, full precision:")
for i, val in enumerate(eigenvalues, start=1):
    print(f"lambda_{i}: {val:.16e}")

print("\nSingular values, full precision:")
for i, val in enumerate(singular_values, start=1):
    print(f"sigma_{i}: {val:.16e}")

loadings = pd.DataFrame(
    pca.components_.T,
    index=[display_label(c) for c in force_constant_cols],
    columns=[f"PC{i}" for i in range(1, 6)],
)

print("\nPC5 loading vector in standardized feature coordinates:")
pc5_loadings = pd.Series(pca.components_[4], index=force_constant_cols)
for col, val in pc5_loadings.items():
    print(f"  {display_label(col):35s}: {val:+.16e}")

loadings


In [ ]:

# -------------------------------------------------------------------
# Summary tables for C
# -------------------------------------------------------------------
C = df_top["C"].to_numpy()
abs_C = np.abs(C)

summary = pd.Series({
    "n": len(C),
    "mean_C": np.mean(C),
    "std_C_ddof0": np.std(C, ddof=0),
    "std_C_ddof1": np.std(C, ddof=1),
    "min_C": np.min(C),
    "q01_C": np.quantile(C, 0.01),
    "q05_C": np.quantile(C, 0.05),
    "median_C": np.median(C),
    "q95_C": np.quantile(C, 0.95),
    "q99_C": np.quantile(C, 0.99),
    "max_C": np.max(C),
    "max_abs_C": np.max(abs_C),
    "median_abs_C": np.median(abs_C),
})

print("C spread summary:")
print(summary.to_string(float_format=lambda x: f"{x:.16e}"))

pc_std = df_top[["PC1", "PC2", "PC3", "PC4", "PC5"]].std(ddof=0)
print("\nScore standard deviations:")
print(pc_std.to_string(float_format=lambda x: f"{x:.16e}"))

ratio = summary["std_C_ddof0"] / pc_std[["PC1", "PC2", "PC3", "PC4"]].mean()
print(f"\nstd(C) / mean[std(PC1..PC4)] = {ratio:.16e}")

# Save tables.
summary.to_csv(OUTPUT_DIR / "constraint_coordinate_C_summary.csv")
loadings.to_csv(OUTPUT_DIR / "pca_loadings_transformed_force_constants.csv")
df_top.to_csv(OUTPUT_DIR / "top5_with_pca_scores_and_C.csv", index=False)
print(f"\nSaved tables to: {OUTPUT_DIR.resolve()}")



## Direct definition of the constraint coordinate

The coordinate \(C\) is computed from standardized variables:

\[
C = w_{\alpha_0}z_{\alpha_0}
+ w_u z_u
+ w_v z_v
+ w_{\alpha_2}z_{\alpha_2}
+ w_{\beta_2}z_{\beta_2},
\]

where

\[
u=\alpha_1+2\beta_1,
\qquad
v=\alpha_1-\beta_1.
\]

Because the variables are standardized before PCA, the coefficients should be interpreted in standardized feature coordinates, not raw force-constant units.


In [ ]:

# -------------------------------------------------------------------
# Histogram of C
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.hist(C, bins=40, edgecolor="black", linewidth=0.8, alpha=0.8)
ax.axvline(0.0, color="black", linewidth=1.0)
ax.set_xlabel(r"$C=s_5$", fontsize=15)
ax.set_ylabel("Count", fontsize=15)
ax.set_title(r"Distribution of the constraint coordinate $C$", fontsize=16, pad=10)
apply_publication_axes(ax)
ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, 3))
fig.tight_layout()
save_figure(fig, "C_histogram")
plt.show()


In [ ]:

# -------------------------------------------------------------------
# Sorted absolute C values on a log scale
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
rank = np.arange(1, len(abs_C) + 1)
ax.semilogy(rank, np.sort(abs_C), marker="o", linestyle="none", markersize=4, alpha=0.75)
ax.set_xlabel("Sorted sample index", fontsize=15)
ax.set_ylabel(r"$|C|$", fontsize=15)
ax.set_title(r"Sorted absolute constraint coordinate", fontsize=16, pad=10)
apply_publication_axes(ax)
fig.tight_layout()
save_figure(fig, "C_abs_sorted_log")
plt.show()


In [ ]:

# -------------------------------------------------------------------
# C versus second-neighbor family coordinate
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
sc = ax.scatter(
    df_top["r_alpha2"], df_top["C"],
    c=df_top[rank_col], s=42, alpha=0.82,
    edgecolors="black", linewidths=0.3,
)
ax.axhline(0.0, color="black", linewidth=1.0)
ax.set_xlabel(r"$r_{\alpha_2}=|\alpha_2|/(|\alpha_2|+|\beta_2|)$", fontsize=15)
ax.set_ylabel(r"$C$", fontsize=15)
ax.set_title(r"Constraint coordinate versus second-neighbor family", fontsize=15, pad=10)
apply_publication_axes(ax)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(display_label(rank_col), fontsize=13)
cbar.ax.tick_params(labelsize=11)
ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 3))
fig.tight_layout()
save_figure(fig, "C_vs_r_alpha2_colored_by_fitness")
plt.show()


In [ ]:

# -------------------------------------------------------------------
# C versus fitness and phonon stability metrics
# -------------------------------------------------------------------
plot_targets = []
for col in [rank_col, fitness1_col, fitness2_col, fitness3_col, minfreq_col, maxfreq_col, numimag_col]:
    if col is not None and col in df_top.columns and col not in plot_targets:
        plot_targets.append(col)

for target in plot_targets:
    valid = df_top[[target, "C", "r_alpha2"]].dropna()
    if valid.empty:
        continue

    fig, ax = plt.subplots(figsize=FIGSIZE)
    sc = ax.scatter(
        valid[target], valid["C"],
        c=valid["r_alpha2"], s=42, alpha=0.82,
        edgecolors="black", linewidths=0.3,
    )
    ax.axhline(0.0, color="black", linewidth=1.0)
    ax.set_xlabel(display_label(target), fontsize=15)
    ax.set_ylabel(r"$C$", fontsize=15)
    ax.set_title(rf"Constraint coordinate versus {display_label(target)}", fontsize=15, pad=10)
    apply_publication_axes(ax)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 3))
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r"$r_{\alpha_2}$", fontsize=13)
    cbar.ax.tick_params(labelsize=11)
    fig.tight_layout()
    safe_target = str(target).replace("/", "_").replace(" ", "_")
    save_figure(fig, f"C_vs_{safe_target}_colored_by_r_alpha2")
    plt.show()


In [ ]:

# -------------------------------------------------------------------
# Correlation table involving C
# -------------------------------------------------------------------
candidate_cols = [
    "C", "abs_C", "r_alpha2", rank_col,
    fitness1_col, fitness2_col, fitness3_col,
    minfreq_col, maxfreq_col, numimag_col,
    mass_col, alat_col,
    alpha0_col, "alpha1_plus_2beta1", "alpha1_minus_beta1", alpha2_col, beta2_col,
]

candidate_cols = [c for c in candidate_cols if c is not None and c in df_top.columns]
analysis_df = df_top[candidate_cols].copy()

pearson_with_C = analysis_df.corr(method="pearson")["C"].sort_values(key=lambda s: s.abs(), ascending=False)
spearman_with_C = analysis_df.corr(method="spearman")["C"].sort_values(key=lambda s: s.abs(), ascending=False)

correlation_summary = pd.DataFrame({
    "pearson_with_C": pearson_with_C,
    "spearman_with_C": spearman_with_C,
})

print("Correlation summary with C:")
print(correlation_summary.to_string(float_format=lambda x: f"{x:+.6f}"))

correlation_summary.to_csv(OUTPUT_DIR / "correlations_with_C.csv")


In [ ]:

# -------------------------------------------------------------------
# PCA map colored by C
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
lim = np.nanmax(np.abs(df_top["C"]))
sc = ax.scatter(
    df_top["PC1"], df_top["PC2"],
    c=df_top["C"], s=46, alpha=0.86,
    edgecolors="black", linewidths=0.25,
    vmin=-lim, vmax=lim,
)
ax.axhline(0.0, color="0.75", linewidth=0.9)
ax.axvline(0.0, color="0.75", linewidth=0.9)
ax.set_xlabel(f"PC1 ({100*explained[0]:.1f}%)", fontsize=15)
ax.set_ylabel(f"PC2 ({100*explained[1]:.1f}%)", fontsize=15)
ax.set_title(r"PCA map colored by constraint coordinate $C$", fontsize=16, pad=10)
apply_publication_axes(ax)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$C$", fontsize=13)
cbar.ax.tick_params(labelsize=11)
fig.tight_layout()
save_figure(fig, "PCA_PC1_PC2_colored_by_C")
plt.show()


In [ ]:

# -------------------------------------------------------------------
# Family-resolved distribution of C without seaborn
# -------------------------------------------------------------------
family_order = [r"$\beta_2$-dominated", "mixed", r"$\alpha_2$-dominated"]
family_data = [df_top.loc[df_top["family"] == fam, "C"].dropna().to_numpy() for fam in family_order]
family_labels = [fam for fam, vals in zip(family_order, family_data) if len(vals) > 0]
family_data = [vals for vals in family_data if len(vals) > 0]

fig, ax = plt.subplots(figsize=FIGSIZE)
positions = np.arange(1, len(family_data) + 1)

ax.boxplot(
    family_data,
    positions=positions,
    widths=0.55,
    showfliers=True,
    patch_artist=False,
    medianprops={"linewidth": 1.4},
)

# Overlay individual points with deterministic jitter.
rng = np.random.default_rng(2026)
for x0, vals in zip(positions, family_data):
    jitter = rng.uniform(-0.10, 0.10, size=len(vals))
    ax.scatter(np.full(len(vals), x0) + jitter, vals, s=28, alpha=0.55, edgecolors="none")

ax.axhline(0.0, color="black", linewidth=1.0)
ax.set_xticks(positions)
ax.set_xticklabels(family_labels, fontsize=12)
ax.set_ylabel(r"$C$", fontsize=15)
ax.set_title(r"Constraint coordinate by second-neighbor family", fontsize=16, pad=10)
apply_publication_axes(ax)
ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 3))
fig.tight_layout()
save_figure(fig, "C_by_second_neighbor_family")
plt.show()


In [ ]:

# -------------------------------------------------------------------
# Optional linear expression for C in standardized coordinates
# -------------------------------------------------------------------
print("C in standardized coordinates:")
terms = []
for col, val in pc5_loadings.items():
    terms.append(f"({val:+.6f}) z[{display_label(col)}]")
print("C = " + " ".join(terms))

# Save the PCA model metadata needed to reproduce C.
metadata = pd.DataFrame({
    "feature": force_constant_cols,
    "display_label": [display_label(c) for c in force_constant_cols],
    "mean_raw": scaler.mean_,
    "scale_raw": scaler.scale_,
    "pc5_loading": pca.components_[4],
})
metadata.to_csv(OUTPUT_DIR / "C_definition_scaler_and_pc5_loadings.csv", index=False)
metadata



## Suggested interpretation

Use the following logic when reading the figures:

- If \(C\) is narrowly centered near zero and uncorrelated with the family coordinate, it is best interpreted as the small thickness of the high-fitness manifold rather than the direction separating the \(\alpha_2\)- and \(\beta_2\)-dominated families.
- If \(C\) correlates with `fitness_norm`, `min_frequency`, or `num_imaginary`, then it may act as a hidden stability coordinate.
- If \(C\) has finite but very small width relative to PC1--PC4, the solutions occupy an approximate four-dimensional hyperplane within the nominal five-dimensional transformed force-constant space.
